# Workshop — 4. Evaluate the Fine-Tuned VLA

Notebook 2 established the zero-shot baseline. Notebook 3 produced an updated checkpoint on SageMaker. This notebook downloads that artifact and evaluates both checkpoints under the same conditions:

- the same SO100 MuJoCo scene;
- the same `top` and `wrist` cameras;
- the same language instruction;
- 400 control steps at 30 Hz;
- the same `placed_in_box` success metric.

Every experimental step is defined in this notebook. The reusable helper in `code/vla_pick.py` remains available for command-line use but is not imported here.

## 1. Install and Configure the Evaluation Runtime

In [ ]:
%pip install -q -r requirements.txt

### Configure the local evaluation process

The renderer and accelerator settings must be applied before importing MuJoCo or PyTorch. On macOS, the FFmpeg library path also enables video encoding and decoding from the notebook process.

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

## 2. Find and Download the SageMaker Artifact

The SageMaker job used uncompressed output, so individual checkpoint files are directly addressable under its model artifact prefix. The next cells find the latest completed job, read `ModelArtifacts.S3ModelArtifacts`, and download only the exported `smolvla/` directory.

In [ ]:
from pathlib import Path

import boto3

TRAINING_JOB_PREFIX = "train-smolvla-so100-sim"
TRAINING_JOB_NAME = None  # Set an exact completed job name to override discovery.


def get_last_job_name(job_name_prefix):
    """Return the newest completed SageMaker Training Job with this prefix."""
    client = boto3.client("sagemaker")
    next_token = None
    while True:
        search_params = {
            "Resource": "TrainingJob",
            "SearchExpression": {
                "Filters": [
                    {"Name": "TrainingJobName", "Operator": "Contains", "Value": job_name_prefix},
                    {"Name": "TrainingJobStatus", "Operator": "Equals", "Value": "Completed"},
                ]
            },
            "SortBy": "CreationTime",
            "SortOrder": "Descending",
            "MaxResults": 100,
        }
        if next_token:
            search_params["NextToken"] = next_token
        response = client.search(**search_params)
        matches = [
            result["TrainingJob"]["TrainingJobName"]
            for result in response["Results"]
            if result["TrainingJob"]["TrainingJobName"].startswith(job_name_prefix)
        ]
        if matches:
            return matches[0]
        next_token = response.get("NextToken")
        if not next_token:
            raise ValueError(f"No completed training jobs found with prefix {job_name_prefix!r}")


sagemaker_client = boto3.client("sagemaker")
training_job_name = TRAINING_JOB_NAME or get_last_job_name(TRAINING_JOB_PREFIX)
job_description = sagemaker_client.describe_training_job(TrainingJobName=training_job_name)
if job_description["TrainingJobStatus"] != "Completed":
    raise RuntimeError(
        f"Training job {training_job_name} is {job_description['TrainingJobStatus']}, not Completed"
    )

artifact_root = job_description["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/")
S3_MODEL_URI = f"{artifact_root}/smolvla/"
LOCAL_MODEL_DIR = (
    Path("outputs/smolvla-so100/evaluations") / training_job_name / "model"
).resolve()

print("Training job:", training_job_name)
print("Remote model:", S3_MODEL_URI)
print("Local model:", LOCAL_MODEL_DIR)

### Materialize the model locally

SageMaker reports the artifact location but does not download it to the notebook. This cell walks the S3 prefix, preserves its directory structure, and ignores SageMaker marker objects.

In [ ]:
from urllib.parse import urlparse


def download_s3_prefix(s3_uri, destination):
    """Download every real object below an S3 prefix into a local directory."""
    parsed = urlparse(s3_uri)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Expected an s3:// URI, got {s3_uri!r}")
    bucket = parsed.netloc
    prefix = parsed.path.lstrip("/")
    if prefix and not prefix.endswith("/"):
        prefix += "/"

    client = boto3.client("s3")
    destination.mkdir(parents=True, exist_ok=True)
    downloaded = []
    for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            key = item["Key"]
            relative = key[len(prefix):]
            if not relative or relative.endswith("/"):
                continue
            if relative.endswith((".sagemaker-uploaded", ".sagemaker-uploading")):
                continue
            local_path = destination / relative
            local_path.parent.mkdir(parents=True, exist_ok=True)
            client.download_file(bucket, key, str(local_path))
            downloaded.append((relative, int(item["Size"])))
    if not downloaded:
        raise FileNotFoundError(f"No model files found under {s3_uri}")
    return downloaded


downloaded = download_s3_prefix(S3_MODEL_URI, LOCAL_MODEL_DIR)
print(f"Downloaded {len(downloaded)} files ({sum(size for _, size in downloaded) / 2**20:.1f} MiB)")
for name, size in downloaded:
    print(f"  {size:>12,}  {name}")

## 3. Validate the Checkpoint Contract

The fine-tuned model must consume the simulation-native cameras and normalization statistics. Notebook 2's degrees/0–100 conversion belongs only to the original real-data checkpoint; the simulation checkpoint uses radians.

In [ ]:
import json

required_files = {
    "config.json",
    "model.safetensors",
    "policy_preprocessor.json",
    "policy_preprocessor_step_5_normalizer_processor.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor_step_0_unnormalizer_processor.safetensors",
    "train_config.json",
    "training_manifest.json",
}
missing = sorted(name for name in required_files if not (LOCAL_MODEL_DIR / name).is_file())
if missing:
    raise FileNotFoundError(f"Incomplete model artifact: {missing}")

config = json.loads((LOCAL_MODEL_DIR / "config.json").read_text())
manifest = json.loads((LOCAL_MODEL_DIR / "training_manifest.json").read_text())
input_features = config["input_features"]

assert config["type"] == "smolvla"
assert input_features["observation.state"]["shape"] == [6]
assert config["output_features"]["action"]["shape"] == [6]
assert {
    key for key, feature in input_features.items() if feature["type"] == "VISUAL"
} == {"observation.images.top", "observation.images.wrist"}
assert manifest["state_action_units"] == "radians"

print(json.dumps({
    "base_model": manifest["base_model"],
    "base_revision": manifest["base_revision"],
    "dataset_episodes": manifest["dataset_episodes"],
    "dataset_frames": manifest["dataset_frames"],
    "camera_keys": manifest["camera_keys"],
    "state_action_units": manifest["state_action_units"],
    "model_size_mib": round((LOCAL_MODEL_DIR / "model.safetensors").stat().st_size / 2**20, 1),
}, indent=2))

## 4. Define the Shared Experiment

The checkpoint is the experimental variable. Scene geometry, camera placement, start pose, language instruction, rollout length, and success metric must be identical. The next cells define that complete contract in the notebook.

In [ ]:
import gc

import mujoco
import numpy as np
import torch
from IPython.display import HTML, Video, display
from strands_robots import Robot
from strands_robots.policies import create_policy
from strands_robots.policies.lerobot_local import clear_model_cache

MODEL_ID = "lucarrr/smolvla_so100_pickplace_finetuned_v2"
MODEL_REVISION = "4fde42badae91b5c88bfe1a399a3f4b994fc0698"
INSTRUCTION = "Pick up the cube and place it in the box."
JOINT_KEYS = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll", "Jaw"]
DATASET_MEAN_STATE = np.array(
    [14.4717, -55.7695, 54.3857, 63.2262, 85.8417, 9.3545],
    dtype=np.float64,
)
GRIPPER_JOINT_RANGE = (-0.175, 1.745)
BOX_CENTER = np.array([0.16, -0.30], dtype=np.float64)
FPS = 30
ROLLOUT_STEPS = 400
BASELINE_VIDEO = Path("smolvla_baseline_eval.mp4").resolve()
FINETUNED_VIDEO = Path("smolvla_finetuned_eval.mp4").resolve()
device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Device:", device)
print("Instruction:", INSTRUCTION)

### Build identical evaluation scenes

These helpers define the common physical experiment. Both checkpoints will receive fresh scenes with exactly the same geometry, cameras, start pose, and instruction.

In [ ]:
def require_success(result, operation):
    """Raise when a strands-robots operation returns an error envelope."""
    if result.get("status") == "success":
        return result
    text = " | ".join(
        str(item.get("text"))
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )
    raise RuntimeError(f"{operation} failed: {text or result}")


def dataset_mean_action():
    """Convert the original real-data mean pose to MuJoCo units."""
    values = np.empty(6, dtype=np.float64)
    values[:5] = np.deg2rad(DATASET_MEAN_STATE[:5])
    jaw_min, jaw_max = GRIPPER_JOINT_RANGE
    values[5] = jaw_min + (DATASET_MEAN_STATE[5] / 100.0) * (jaw_max - jaw_min)
    return dict(zip(JOINT_KEYS, values.tolist(), strict=True))


def build_scene():
    """Build one fresh SO100 pick-place experiment."""
    sim = Robot("so100", mesh=False)
    require_success(
        sim.add_object(
            name="cube",
            shape="box",
            position=[0.0, -0.35, 0.015],
            size=[0.03, 0.03, 0.03],
            color=[0.9, 0.12, 0.08, 1.0],
            mass=0.03,
        ),
        "add cube",
    )
    blue = [0.12, 0.28, 0.75, 1.0]
    x, y = BOX_CENTER
    parts = {
        "target_base": ([x, y, 0.005], [0.12, 0.12, 0.01]),
        "target_left": ([x - 0.055, y, 0.025], [0.01, 0.12, 0.05]),
        "target_right": ([x + 0.055, y, 0.025], [0.01, 0.12, 0.05]),
        "target_back": ([x, y + 0.055, 0.025], [0.10, 0.01, 0.05]),
        "target_front": ([x, y - 0.055, 0.025], [0.10, 0.01, 0.05]),
    }
    for name, (position, size) in parts.items():
        require_success(
            sim.add_object(
                name=name,
                shape="box",
                position=list(position),
                size=list(size),
                color=blue,
                is_static=True,
            ),
            f"add {name}",
        )
    require_success(
        sim.add_camera(
            name="top",
            position=[0.45, -0.60, 0.42],
            target=[0.06, -0.31, 0.06],
            fov=55,
            width=640,
            height=480,
        ),
        "add top camera",
    )
    require_success(
        sim.add_camera(
            name="wrist",
            position=[0.02, -0.56, 0.18],
            target=[0.04, -0.31, 0.04],
            fov=70,
            width=640,
            height=480,
        ),
        "add wrist camera",
    )
    require_success(
        sim.send_action(dataset_mean_action(), robot_name="so100", n_substeps=600),
        "move SO100 to the dataset-mean pose",
    )
    return sim

### Separate execution from task success

`run_rollout()` owns the shared policy-execution settings. `task_diagnostics()` independently measures the cube position, so a completed API call cannot be mistaken for a successful manipulation.

In [ ]:
def task_diagnostics(sim):
    """Measure the cube pose and the geometric placed-in-box condition."""
    cube_id = mujoco.mj_name2id(sim.mj_model, mujoco.mjtObj.mjOBJ_BODY, "cube")
    position = np.asarray(sim.mj_data.xpos[cube_id], dtype=float).copy()
    inside_box_xy = bool(
        np.all(np.abs(position[:2] - BOX_CENTER) < np.array([0.045, 0.045]))
    )
    return {
        "cube_position_m": np.round(position, 4).tolist(),
        "placed_in_box": inside_box_xy and position[2] < 0.10,
    }


def run_rollout(sim, policy, video_path):
    """Execute the common 400-step policy experiment and record its video."""
    return sim.run_policy(
        robot_name="so100",
        policy_object=policy,
        instruction=INSTRUCTION,
        n_steps=ROLLOUT_STEPS,
        control_frequency=FPS,
        fast_mode=True,
        video={"path": str(video_path), "camera": "top", "fps": FPS},
    )

## 5. Reproduce the Zero-Shot Baseline

The original checkpoint was trained on real SO100 data. Its embodiment adapter maps `wrist`/`top` to `camera1`/`camera2` and converts MuJoCo joint values to the checkpoint's degrees and 0–100 gripper convention.

In [ ]:
baseline_embodiment = {
    "name": "so100_smolvla_pickplace_sim",
    "obs_rename": {
        "wrist": "observation.images.camera1",
        "top": "observation.images.camera2",
    },
    "state_keys": JOINT_KEYS,
    "action_keys": JOINT_KEYS,
    "dim_policy": "strict",
    "state_units": "degrees",
    "action_units": "degrees",
    "gripper_index": 5,
    "gripper_joint_range": list(GRIPPER_JOINT_RANGE),
}

baseline_policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path=MODEL_ID,
    revision=MODEL_REVISION,
    policy_type="smolvla",
    device=device,
    embodiment=baseline_embodiment,
    strict_keys=True,
)
baseline_sim = build_scene()
baseline_result = run_rollout(baseline_sim, baseline_policy, BASELINE_VIDEO)
baseline_diagnostics = task_diagnostics(baseline_sim)

print("Checkpoint:", f"{MODEL_ID}@{MODEL_REVISION[:8]}")
print("run_policy status:", baseline_result["status"])
print("Task diagnostics:", baseline_diagnostics)

### Release the baseline before loading the adapted model

Only one 450M-parameter checkpoint should remain resident on a laptop. This cell removes Python references, clears the `lerobot_local` model cache, and releases accelerator memory.

In [ ]:
del baseline_policy
del baseline_sim
released_models = clear_model_cache()
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Released {released_models} cached model(s) from memory.")

## 6. Evaluate the Fine-Tuned Checkpoint

The fine-tuned checkpoint was trained directly on simulation-native `top` and `wrist` features. Its state, actions, and exported normalization statistics are in radians, so it must not reuse the baseline unit conversion.

In [ ]:
finetuned_embodiment = {
    "name": "so100_smolvla_sim_finetuned",
    "obs_rename": {
        "wrist": "observation.images.wrist",
        "top": "observation.images.top",
    },
    "state_keys": JOINT_KEYS,
    "action_keys": JOINT_KEYS,
    "dim_policy": "strict",
    "state_units": "radians",
    "action_units": "radians",
}

finetuned_policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path=str(LOCAL_MODEL_DIR),
    policy_type="smolvla",
    device=device,
    embodiment=finetuned_embodiment,
    strict_keys=True,
)
finetuned_sim = build_scene()
finetuned_result = run_rollout(finetuned_sim, finetuned_policy, FINETUNED_VIDEO)
finetuned_diagnostics = task_diagnostics(finetuned_sim)

print("Checkpoint:", LOCAL_MODEL_DIR)
print("run_policy status:", finetuned_result["status"])
print("Task diagnostics:", finetuned_diagnostics)

## 7. Compare Before and After

`run_status` measures whether the software loop executed. `placed_in_box` measures whether the robot completed the physical task.

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "checkpoint": "zero-shot real-data checkpoint",
        "run_status": baseline_result["status"],
        "cube_position_m": baseline_diagnostics["cube_position_m"],
        "placed_in_box": bool(baseline_diagnostics["placed_in_box"]),
        "video": str(BASELINE_VIDEO),
    },
    {
        "checkpoint": "simulation fine-tuned checkpoint",
        "run_status": finetuned_result["status"],
        "cube_position_m": finetuned_diagnostics["cube_position_m"],
        "placed_in_box": bool(finetuned_diagnostics["placed_in_box"]),
        "video": str(FINETUNED_VIDEO),
    },
])
display(comparison)

display(HTML("<h3>Zero-shot baseline</h3>"))
display(Video(str(BASELINE_VIDEO), embed=True, width=640))
display(HTML("<h3>Simulation fine-tuned</h3>"))
display(Video(str(FINETUNED_VIDEO), embed=True, width=640))

## Interpretation

A successful fine-tuned rollout closes the workshop loop: matching demonstrations can correct the visual and embodiment distribution shift observed in Notebook 2.

A failed rollout does not invalidate the training job. Eight episodes validate the engineering pipeline but remain a very small learning dataset. Inspect the video and cube trajectory, then expand the successful demonstration set before changing model architecture.